In [ ]:
from secfsn_plus_models.notebook_entry import init_notebook
init_notebook()


# Baseline Model Training

This notebook trains the baseline linear regression model on the demo SECFSN+ dataset and checks whether the resulting predictions carry cross-sectional signal.

It is designed to answer two questions:

- Does the model produce signal?
- Is IC stable across periods?


## 1. Load Data

Load the demo dataset using the repository's notebook artifacts helper.


In [ ]:
from secfsn_plus_models.notebook_utils import load_notebook_demo_artifacts


demo = load_notebook_demo_artifacts(
    model_type="linear_regression",
    train_end_period="2020Q4",
)

X = demo.X
y = demo.y

print(f"Dataset source: {demo.dataset_source}")
print(f"Feature matrix shape: {X.shape}")
print(f"Label series shape: {y.shape}")


## 2. Train Model

Split the aligned dataset by period, then fit the baseline linear regression model and generate out-of-sample predictions.


In [ ]:
from secfsn_plus_models.models.registry import ModelRegistry
from secfsn_plus_models.splits import time_based_split
from secfsn_plus_models.trainer import train_and_predict


X_train, y_train, _, _, X_test, y_test = time_based_split(
    X,
    y,
    train_end_period="2020Q4",
)

model = ModelRegistry.create("linear_regression")
predictions = train_and_predict(model, X_train, y_train, X_test)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Training periods: {X_train.index.get_level_values('period').nunique()}")
print(f"Test periods: {X_test.index.get_level_values('period').nunique()}")
predictions.head()


## 3. Predictions

Align the test-set predictions with the realized target values so later evaluation uses the same rows in the same order.


In [ ]:
from secfsn_plus_models.utils import copy_aligned_series_pair


y_aligned, predictions_aligned = copy_aligned_series_pair(
    y_test,
    predictions,
    object_names=("y_test", "predictions"),
    require_non_empty=True,
)

print(f"Aligned rows: {len(y_aligned)}")
print(f"Indexes match: {y_aligned.index.equals(predictions_aligned.index)}")

aligned_preview = (
    y_aligned.rename("actual")
    .to_frame()
    .assign(prediction=predictions_aligned)
    .reset_index()
)

aligned_preview.head(10)


## 4. Evaluation

Compute IC and RankIC across periods and summarize their mean values.


In [ ]:
from secfsn_plus_models.metrics import information_coefficient, rank_information_coefficient


ic_by_period = information_coefficient(y_aligned, predictions_aligned)
rank_ic_by_period = rank_information_coefficient(y_aligned, predictions_aligned)

print(f"IC mean: {ic_by_period.mean():.4f}")
print(f"RankIC mean: {rank_ic_by_period.mean():.4f}")


## 5. IC by Period

Plot the period-by-period IC series to see whether signal is stable over time.


In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 4))
plt.plot(ic_by_period.index.astype(str), ic_by_period.values)
plt.axhline(0.0)
plt.title("IC by Period")
plt.xlabel("Period")
plt.ylabel("IC")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Prediction Distribution

Review the distribution of model predictions.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(predictions_aligned.values, bins=20)
plt.title("Prediction Distribution")
plt.xlabel("Prediction")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 7. Predictions vs Actual

Compare predictions against realized outcomes with a scatter plot.


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(predictions_aligned.values, y_aligned.values)
plt.title("Predictions vs Actual")
plt.xlabel("Prediction")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## Takeaways

- Positive IC and RankIC means indicate whether the baseline linear regression produces usable signal on the demo test periods.
- The IC-by-period chart helps assess whether that signal is stable or concentrated in only a few periods.
